In [1]:
import os
import numpy as np
import joblib

from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn

from torch.utils.tensorboard import SummaryWriter

In [2]:
save_dir = "data/bike_processed"

# --- Load arrays ---
raw_data    = np.load(os.path.join(save_dir, "raw_data.npy"))
counts = np.load(os.path.join(save_dir, "counts.npy"))
count_stats = np.load(os.path.join(save_dir, "count_stats.npy"))
train_idx   = np.load(os.path.join(save_dir, "train_idx.npy"))
val_idx     = np.load(os.path.join(save_dir, "val_idx.npy"))
test_idx    = np.load(os.path.join(save_dir, "test_idx.npy"))
naive_mae   = np.load(os.path.join(save_dir, "naive_mae.npy"))
preprocessor = joblib.load(os.path.join(save_dir, "preprocessor_rnn.pkl"))

# --- Recover split sizes ---
num_train = len(train_idx)
num_val   = len(val_idx)
num_test  = len(test_idx)

# --- Count stats in the training set ---
count_mean  = count_stats[0]
count_std   = count_stats[1]

# --- Sanity check ---
print(f"raw_data shape:    {raw_data.shape}")
print(f"counts shape: {counts.shape}")
print(f"train/val/test:    {num_train} / {num_val} / {num_test}")
print(f"counts range: {counts.min():.0f} to {counts.max():.0f} bikes/hour")
print(f"\nNaive baseline — Val MAE: {naive_mae[0]:.2f} | Test MAE: {naive_mae[1]:.2f} bikes/hour")

raw_data shape:    (17210, 28)
counts shape: (17210,)
train/val/test:    12047 / 2581 / 2582
counts range: 1 to 977 bikes/hour

Naive baseline — Val MAE: 93.01 | Test MAE: 79.18 bikes/hour


In [3]:
count_norm = (counts - count_mean) / count_std

In [4]:
class TimeseriesDataset(Dataset):
    def __init__(self, data, targets, sequence_length, sampling_rate, 
                 start_index, end_index, shuffle=False):
        self.data            = data
        self.targets         = targets
        self.sequence_length = sequence_length
        self.sampling_rate   = sampling_rate

        # Valid starting indices: each sequence of length sequence_length
        # sampled every sampling_rate steps spans `span` rows, and the
        # target sits `delay` rows past the start. Clamp end_index so no
        # window ever reads past the end of the array.
        span      = (sequence_length - 1) * sampling_rate
        max_start = len(data) - span - delay - 1
        safe_end  = min(end_index, max_start + 1)
        self.indices = np.arange(start_index, safe_end)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        start = self.indices[idx]
        # Sample every `sampling_rate` steps for `sequence_length` steps
        steps = np.arange(start, start + self.sequence_length * self.sampling_rate, 
                          self.sampling_rate)
        x = self.data[steps]
        # Target is `delay` steps ahead of the sequence start
        y = self.targets[start + delay]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

In [5]:
sampling_rate   = 1    # already hourly
sequence_length = 168  # look back 168 hours = 1 full week
delay           = 1    # predict 1 hour ahead
batch_size      = 256

train_dataset = TimeseriesDataset(
    data=raw_data, targets=count_norm,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=0, end_index=num_train,
)
val_dataset = TimeseriesDataset(
    data=raw_data, targets=count_norm,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=num_train, end_index=num_train + num_val,
)
test_dataset = TimeseriesDataset(
    data=raw_data, targets=count_norm,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=num_train + num_val, end_index=len(raw_data) - delay,
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

# --- Sanity check ---
inputs, targets = next(iter(train_loader))
print(f"Input shape:  {inputs.shape}")   # (256, 168, num_features)
print(f"Target shape: {targets.shape}")  # (256,)
print(f"Target range: {targets.min():.0f} to {targets.max():.0f}")

Input shape:  torch.Size([256, 168, 28])
Target shape: torch.Size([256])
Target range: -1 to 3


## 1. Model — GRU

We start simple, exactly as the tips recommend: **a single GRU layer with 64
hidden units** followed by a linear head that maps the last hidden state to a
single regression output (the normalized bike count one hour ahead).

- The GRU consumes the full 24-hour sequence `(batch, seq_len, num_features)`
  and returns an output for every timestep. We keep only the **last timestep's
  hidden state**, which summarizes the whole window, and feed it to the head.
- `num_layers`, `dropout` and `bidirectional` are exposed as arguments so the
  model can be made more complex later *only if* the train/val gap shows the
  simple version is under- or over-fitting. Dropout is applied between stacked
  GRU layers (PyTorch only applies it when `num_layers > 1`) and again before
  the linear head.

In [6]:
class GRURegressor(nn.Module):
    """Single-output regressor built on a GRU encoder.

    Args:
        input_size:   number of input features per timestep (28 here).
        hidden_size:  GRU hidden units (start with 64).
        num_layers:   number of stacked GRU layers (start with 1).
        dropout:      dropout prob between stacked layers and before the head.
        bidirectional: whether to read the sequence in both directions.
    """
    def __init__(self, input_size, hidden_size=64, num_layers=1,
                 dropout=0.0, bidirectional=False):
        super().__init__()
        self.num_layers    = num_layers
        self.hidden_size   = hidden_size
        self.bidirectional = bidirectional

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,                 # input is (batch, seq, feature)
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )

        num_directions = 2 if bidirectional else 1
        self.dropout = nn.Dropout(dropout)
        self.head    = nn.Linear(hidden_size * num_directions, 1)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        out, h_n = self.gru(x)              # out: (batch, seq_len, hidden*dirs)

        if self.bidirectional:
            # concat the last forward state and the last backward state
            h_fwd = h_n[-2]
            h_bwd = h_n[-1]
            last = torch.cat([h_fwd, h_bwd], dim=1)
        else:
            last = h_n[-1]                  # (batch, hidden) — top layer state

        last = self.dropout(last)
        y = self.head(last)                # (batch, 1)
        return y.squeeze(1)                # (batch,) to match target shape

In [7]:
# --- Device + model instantiation ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

num_features = raw_data.shape[1]           # 28

# We started from the simple 1-layer/64-unit model (it already beats the
# naive baseline by a wide margin). A short sweep over window length and
# capacity showed the best generalization comes from feeding a FULL WEEK of
# history (sequence_length=168) to a larger, bidirectional, regularized GRU:
#   - 2 stacked layers, hidden_size=128  -> enough capacity for the weekly pattern
#   - dropout 0.4                         -> controls the train/val gap
#   - bidirectional=True                  -> reads the week forwards and backwards
# This brings val/test MAE to ~44/46 bikes/hour. Going much lower is hard: as
# the tips note, the sklearn tree baseline (~34) is very strong on this data.
model = GRURegressor(
    input_size=num_features,
    hidden_size=128,
    num_layers=2,
    dropout=0.4,
    bidirectional=True,
).to(device)

print(model)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

Using device: cuda
GRURegressor(
  (gru): GRU(28, 128, num_layers=2, batch_first=True, dropout=0.4, bidirectional=True)
  (dropout): Dropout(p=0.4, inplace=False)
  (head): Linear(in_features=256, out_features=1, bias=True)
)
Trainable parameters: 418,049


## 2. Training setup

- **Loss:** MSE on the *normalized* targets. Training on the normalized scale
  keeps gradients well-conditioned; we convert back to bikes/hour only when we
  report MAE.
- **Optimizer:** Adam, `lr=1e-3` (drop to `1e-4` if the loss stalls early).
- **ReduceLROnPlateau:** halves the LR when val MAE stops improving, so we don't
  have to tune the schedule by hand.
- **ModelCheckpoint (manual):** every time val MAE hits a new best we save the
  weights to `best_model.pt`. Training can safely run long — the best point is
  never lost.
- **Early stopping (manual):** if val MAE does not improve for `patience`
  epochs we stop.
- **TensorBoard:** train/val loss and MAE are logged to `runs/` so the curves
  can be watched live with `tensorboard --logdir runs`.

Because the targets are normalized, **MAE in bikes/hour = MAE_normalized ×
`count_std`** (the mean shift cancels inside an absolute difference). That is
the single conversion used everywhere below.

In [8]:
from torch.optim.lr_scheduler import ReduceLROnPlateau
import datetime

# --- Hyperparameters ---
num_epochs    = 150     # run long; early stopping decides when to stop
learning_rate = 1e-3
patience      = 20      # early-stopping patience (epochs w/o val improvement)
ckpt_path     = "best_model.pt"

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

# TensorBoard writer with a unique run name per launch
run_name = datetime.datetime.now().strftime("gru_%Y%m%d-%H%M%S")
writer   = SummaryWriter(os.path.join("runs", run_name))
print(f"Logging to runs/{run_name}")

Logging to runs/gru_20261022-163546


In [9]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """Return (mean MSE loss, MAE in bikes/hour) over a loader."""
    model.eval()
    total_loss = 0.0
    total_abs  = 0.0        # sum of |error| in normalized units
    n_samples  = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        loss = criterion(pred, y)

        bs = y.size(0)
        total_loss += loss.item() * bs
        total_abs  += torch.sum(torch.abs(pred - y)).item()
        n_samples  += bs

    mean_loss = total_loss / n_samples
    mae_norm  = total_abs / n_samples
    mae_bikes = mae_norm * count_std        # convert to bikes/hour
    return mean_loss, mae_bikes

In [10]:
# --- Training loop ---
best_val_mae = float("inf")
epochs_no_improve = 0

for epoch in range(1, num_epochs + 1):
    # ---- train ----
    model.train()
    running_loss = 0.0
    running_abs  = 0.0
    n_seen       = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()

        bs = y.size(0)
        running_loss += loss.item() * bs
        running_abs  += torch.sum(torch.abs(pred.detach() - y)).item()
        n_seen       += bs

    train_loss = running_loss / n_seen
    train_mae  = (running_abs / n_seen) * count_std

    # ---- validate ----
    val_loss, val_mae = evaluate(model, val_loader, criterion, device)

    # ---- scheduler on val MAE ----
    scheduler.step(val_mae)
    current_lr = optimizer.param_groups[0]["lr"]

    # ---- TensorBoard logging ----
    writer.add_scalar("Loss/train", train_loss, epoch)
    writer.add_scalar("Loss/val",   val_loss,   epoch)
    writer.add_scalar("MAE/train",  train_mae,  epoch)
    writer.add_scalar("MAE/val",    val_mae,    epoch)
    writer.add_scalar("LR",         current_lr, epoch)

    # ---- checkpoint best model ----
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        epochs_no_improve = 0
        torch.save(model.state_dict(), ckpt_path)
        flag = "  <-- best, saved"
    else:
        epochs_no_improve += 1
        flag = ""

    print(f"Epoch {epoch:3d}/{num_epochs} | "
          f"train MAE {train_mae:6.2f} | val MAE {val_mae:6.2f} | "
          f"lr {current_lr:.1e}{flag}")

    # ---- early stopping ----
    if epochs_no_improve >= patience:
        print(f"\nEarly stopping at epoch {epoch} "
              f"(no val improvement for {patience} epochs).")
        break

writer.close()
print(f"\nBest val MAE: {best_val_mae:.2f} bikes/hour  (weights in {ckpt_path})")

Epoch   1/150 | train MAE  76.36 | val MAE  98.99 | lr 1.0e-03  <-- best, saved


Epoch   2/150 | train MAE  55.60 | val MAE  70.39 | lr 1.0e-03  <-- best, saved


Epoch   3/150 | train MAE  41.32 | val MAE  54.81 | lr 1.0e-03  <-- best, saved


Epoch   4/150 | train MAE  36.46 | val MAE  52.03 | lr 1.0e-03  <-- best, saved


Epoch   5/150 | train MAE  34.07 | val MAE  47.05 | lr 1.0e-03  <-- best, saved


...

Epoch  36/150 | train MAE  21.70 | val MAE  49.21 | lr 1.3e-04


Epoch  37/150 | train MAE  21.63 | val MAE  48.04 | lr 1.3e-04


Epoch  38/150 | train MAE  21.58 | val MAE  49.46 | lr 6.3e-05


Epoch  39/150 | train MAE  21.68 | val MAE  47.06 | lr 6.3e-05


Epoch  40/150 | train MAE  21.54 | val MAE  48.06 | lr 6.3e-05

Early stopping at epoch 40 (no val improvement for 20 epochs).

Best val MAE: 42.05 bikes/hour  (weights in best_model.pt)


## 3. Evaluation

Reload the **best** checkpoint (not the last epoch, which may have overfit) and
report MAE in bikes/hour on validation and test. We compare against:

- the **naive baseline** loaded earlier, and
- the target of being **within 10 bikes/hour of the sklearn baseline
  (< 44 bikes/hour)**, which the tips call an excellent result.

In [11]:
# --- Load the best checkpoint and evaluate ---
model.load_state_dict(torch.load(ckpt_path, map_location=device))

val_loss,  val_mae  = evaluate(model, val_loader,  criterion, device)
test_loss, test_mae = evaluate(model, test_loader, criterion, device)

print("=" * 52)
print(f"{'Model':<22}{'Val MAE':>14}{'Test MAE':>14}")
print("-" * 52)
print(f"{'Naive baseline':<22}{naive_mae[0]:>11.2f}   {naive_mae[1]:>11.2f}")
print(f"{'GRU (ours)':<22}{val_mae:>11.2f}   {test_mae:>11.2f}")
print("=" * 52)

print(f"\nImprovement over naive (test): "
      f"{naive_mae[1] - test_mae:+.2f} bikes/hour")
if test_mae < 44:
    print("Test MAE < 44 bikes/hour — excellent result (within 10 of sklearn).")
else:
    print("Test MAE >= 44 — try more epochs, tune LR, or add light regularization.")

Model                        Val MAE      Test MAE
----------------------------------------------------
Naive baseline              93.01         79.18
GRU (ours)                  42.05         44.24

Improvement over naive (test): +34.94 bikes/hour
Test MAE >= 44 — try more epochs, tune LR, or add light regularization.
